In [5]:
#IQR Method
import pandas as pd
import numpy as np

# Step 1: Load the dataset
df = pd.read_csv("AirQualityUCI.csv", sep=";", decimal=",")

print("Original Dataset:")
print(df.head())

# Step 2: Select numerical columns
numeric_cols = df.select_dtypes(include=np.number).columns

# Remove columns that contain only missing values
numeric_cols = [
    col for col in numeric_cols
    if df[col].notna().any()
]

# Step 3: Handle missing values
df[numeric_cols] = df[numeric_cols].replace(-200, np.nan)

# Step 4: Detect and treat outliers using IQR
df_treated = df.copy()

for col in numeric_cols:

    # Calculate Q1 and Q3
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    # Calculate IQR
    IQR = Q3 - Q1

    # Calculate lower and upper limits
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    # Identify outliers
    outliers = df[
        (df[col] < lower_limit) |
        (df[col] > upper_limit)
    ]

    print("\nColumn:", col)
    print("Lower Limit:", lower_limit)
    print("Upper Limit:", upper_limit)
    print("Number of Outliers:", len(outliers))

    # Treat outliers using capping
    df_treated[col] = df[col].clip(
        lower=lower_limit,
        upper=upper_limit
    )

# Step 5: Display treated dataset
print("\nDataset After IQR Treatment:")
print(df_treated.head())

# Step 6: Save the treated dataset
df_treated.to_csv("AirQuality_IQR_Treated.csv", index=False)

print("\nTreated dataset saved successfully.")


Original Dataset:
         Date      Time  CO(GT)  PT08.S1(CO)  NMHC(GT)  C6H6(GT)  \
0  10/03/2004  18.00.00     2.6       1360.0     150.0      11.9   
1  10/03/2004  19.00.00     2.0       1292.0     112.0       9.4   
2  10/03/2004  20.00.00     2.2       1402.0      88.0       9.0   
3  10/03/2004  21.00.00     2.2       1376.0      80.0       9.2   
4  10/03/2004  22.00.00     1.6       1272.0      51.0       6.5   

   PT08.S2(NMHC)  NOx(GT)  PT08.S3(NOx)  NO2(GT)  PT08.S4(NO2)  PT08.S5(O3)  \
0         1046.0    166.0        1056.0    113.0        1692.0       1268.0   
1          955.0    103.0        1174.0     92.0        1559.0        972.0   
2          939.0    131.0        1140.0    114.0        1555.0       1074.0   
3          948.0    172.0        1092.0    122.0        1584.0       1203.0   
4          836.0    131.0        1205.0    116.0        1490.0       1110.0   

      T    RH      AH  Unnamed: 15  Unnamed: 16  
0  13.6  48.9  0.7578          NaN          NaN 

In [4]:
#Z-score Method
import pandas as pd
import numpy as np
from scipy.stats import zscore

# Step 1: Load the dataset
df = pd.read_csv("AirQualityUCI.csv", sep=";", decimal=",")

print("Original Dataset:")
print(df.head())

# Step 2: Select numerical columns
numeric_cols = df.select_dtypes(include=np.number).columns

# Step 3: Replace missing value code with NaN
df[numeric_cols] = df[numeric_cols].replace(-200, np.nan)

# Step 4: Create a copy of the dataset
df_treated = df.copy()

# Step 5: Detect and treat outliers using Z-score
for col in numeric_cols:

    # Skip columns with no valid values or zero variation
    if df[col].count() < 2 or df[col].std() == 0:
        continue

    # Calculate Z-score
    z_scores = zscore(df[col], nan_policy="omit", ddof=1)

    # Identify outliers where absolute Z-score > 3
    outlier_mask = pd.Series(
        np.abs(z_scores) > 3,
        index=df.index
    ).fillna(False)

    print("\nColumn:", col)
    print("Number of Outliers:", outlier_mask.sum())

    # Calculate treatment limits
    mean = df[col].mean()
    std = df[col].std()

    lower_limit = mean - 3 * std
    upper_limit = mean + 3 * std

    # Treat outliers using capping
    df_treated[col] = df[col].clip(
        lower=lower_limit,
        upper=upper_limit
    )

# Step 6: Display treated dataset
print("\nDataset After Z-score Treatment:")
print(df_treated.head())

# Step 7: Save the treated dataset
df_treated.to_csv("AirQuality_ZScore_Treated.csv", index=False)

print("\nTreated dataset saved successfully.")


Original Dataset:
         Date      Time  CO(GT)  PT08.S1(CO)  NMHC(GT)  C6H6(GT)  \
0  10/03/2004  18.00.00     2.6       1360.0     150.0      11.9   
1  10/03/2004  19.00.00     2.0       1292.0     112.0       9.4   
2  10/03/2004  20.00.00     2.2       1402.0      88.0       9.0   
3  10/03/2004  21.00.00     2.2       1376.0      80.0       9.2   
4  10/03/2004  22.00.00     1.6       1272.0      51.0       6.5   

   PT08.S2(NMHC)  NOx(GT)  PT08.S3(NOx)  NO2(GT)  PT08.S4(NO2)  PT08.S5(O3)  \
0         1046.0    166.0        1056.0    113.0        1692.0       1268.0   
1          955.0    103.0        1174.0     92.0        1559.0        972.0   
2          939.0    131.0        1140.0    114.0        1555.0       1074.0   
3          948.0    172.0        1092.0    122.0        1584.0       1203.0   
4          836.0    131.0        1205.0    116.0        1490.0       1110.0   

      T    RH      AH  Unnamed: 15  Unnamed: 16  
0  13.6  48.9  0.7578          NaN          NaN 